In [2]:
# Cell 1 — Imports and Configuration
import os
import json
import yaml
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.merge import merge
from shapely.geometry import box, mapping
import urllib.request
import warnings
warnings.filterwarnings('ignore')

# Load config
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

DISTRICT = config['study_area']['district']
BBOX = config['study_area']['bbox']
MIN_LON = BBOX['min_lon']
MAX_LON = BBOX['max_lon']
MIN_LAT = BBOX['min_lat']
MAX_LAT = BBOX['max_lat']

# Directory setup
RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'
BUNDLED_DIR = '../data/bundled/'
for d in [RAW_DIR, PROCESSED_DIR, BUNDLED_DIR,
          '../outputs/static/', '../outputs/live/']:
    os.makedirs(d, exist_ok=True)

# ── CONFIG FIX: ensure config.yaml has correct Netrokona bbox ──
# Netrokona: lat 24.20–25.10, lon 90.25–91.25
# If your config is pointing to Feni, update it here:
EXPECTED_BBOX = {
    'min_lon': 90.25, 'max_lon': 91.25,
    'min_lat': 24.20, 'max_lat': 25.10
}
if abs(MIN_LAT - 24.20) > 0.5:
    print("⚠️  WARNING: BBOX looks wrong for Netrokona. Overriding.")
    MIN_LON, MAX_LON = 90.25, 91.25
    MIN_LAT, MAX_LAT = 24.20, 25.10
    BBOX = EXPECTED_BBOX

print(f"✅ Configuration loaded")
print(f"   District : {DISTRICT}")
print(f"   Bbox     : {MIN_LON},{MIN_LAT} → {MAX_LON},{MAX_LAT}")
print(f"   Dirs ready")

✅ Configuration loaded
   District : Netrokona
   Bbox     : 90.25,24.2 → 91.25,25.1
   Dirs ready


In [3]:
## CELL 2 — Download Union Boundaries

# Cell 2 — Download Netrokona Union Boundaries (GADM Level 4)
print("Downloading Netrokona union boundaries...")

import urllib.request

unions_path = f'{PROCESSED_DIR}netrokona_boundaries.geojson'

if os.path.exists(unions_path):
    unions = gpd.read_file(unions_path)
    if len(unions) > 20:           # real union data has ~92 features
        print(f"✅ Cached boundaries loaded: {len(unions)} unions")
    else:
        os.remove(unions_path)     # bad cached file, re-download
        unions = None
else:
    unions = None

if unions is None:
    # ── Strategy 1: GADM 4.1 Level 4 (Union level) ──────────────
    gadm_url = (
        "https://geodata.ucdavis.edu/gadm/gadm4.1/json/"
        "gadm41_BGD_4.json"
    )
    print("  Downloading GADM Level 4 for Bangladesh (~50 MB)...")
    try:
        all_bd = gpd.read_file(gadm_url)

        # GADM spells it 'Netrakona' — catch both spellings
        mask_district = all_bd['NAME_2'].str.contains(
            'Netrakona|Netrokona', case=False, na=False
        )
        unions = all_bd[mask_district].copy()
        unions['union_name']    = unions['NAME_4']
        unions['upazila_name']  = unions['NAME_3']
        unions['district_name'] = unions['NAME_2']
        unions = unions.to_crs('EPSG:4326')
        print(f"  Found {len(unions)} unions from GADM")

    except Exception as e:
        print(f"  GADM failed: {e}")
        # ── Strategy 2: GeoBoundaries ADM4 ──────────────────────
        try:
            print("  Trying GeoBoundaries ADM4...")
            url = "https://www.geoboundaries.org/api/current/gbOpen/BGD/ADM4/"
            r = requests.get(url, timeout=30).json()
            gdf_all = gpd.read_file(r['gjDownloadURL'])
            unions = gdf_all[
                gdf_all['shapeName'].str.contains(
                    'Netrakona|Netrokona', case=False, na=False
                )
            ].copy()
            unions['union_name']   = unions['shapeName']
            unions['upazila_name'] = unions.get('shapeGroup', 'Unknown')
            print(f"  Found {len(unions)} unions from GeoBoundaries")
        except Exception as e2:
            print(f"  GeoBoundaries also failed: {e2}")
            print("  Using bbox fallback — REPLACE with real data before submission")
            bbox_geom = box(MIN_LON, MIN_LAT, MAX_LON, MAX_LAT)
            unions = gpd.GeoDataFrame(
                {'union_name': ['Netrokona'], 'upazila_name': ['Sadar'],
                 'geometry': [bbox_geom]},
                crs='EPSG:4326'
            )

    unions['union_id'] = range(len(unions))
    unions.to_file(unions_path, driver='GeoJSON')
    print(f"✅ Boundaries saved: {len(unions)} unions")

print(f"   Columns: {list(unions.columns)}")
print(unions[['union_name','upazila_name']].head())

  Found 92 unions from GADM
✅ Boundaries saved: 92 unions
   Columns: ['GID_4', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'GID_2', 'NAME_2', 'GID_3', 'NAME_3', 'NAME_4', 'VARNAME_4', 'TYPE_4', 'ENGTYPE_4', 'CC_4', 'geometry', 'union_name', 'upazila_name', 'district_name', 'union_id']
     union_name upazila_name
3412   Baniajan       Atpara
3413       Duaz       Atpara
3414  Loneshwar       Atpara
3415   Sarmaisa       Atpara
3416      Sonai       Atpara


In [6]:
# Cell 3 — WorldPop Population (Optimized for Speed)
import os
import urllib.request
import rasterio
from rasterio.mask import mask
from shapely.geometry import box, mapping
import numpy as np

print("🚀 Starting WorldPop Download & Process...")

# ── 3a. Total population ─────────────────────────────────────────
worldpop_url = "https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/BGD/bgd_ppp_2020_UNadj.tif"
raw_pop_path  = f"{RAW_DIR}bgd_population_2020.tif"
clip_pop_path = f"{PROCESSED_DIR}netrokona_population.tif"

if not os.path.exists(raw_pop_path):
    print("  📥 Downloading main population raster (~150 MB)...")
    def _progress(b, bs, ts):
        if b % 1000 == 0:
            print(f"    {min(b*bs/ts*100,100):.1f}% complete", end='\r')
    urllib.request.urlretrieve(worldpop_url, raw_pop_path, reporthook=_progress)
    print("\n  ✅ Main download complete")
else:
    print("  📦 Population raster found in cache.")

# Clip to Netrokona
bbox_geom = [mapping(box(MIN_LON, MIN_LAT, MAX_LON, MAX_LAT))]
with rasterio.open(raw_pop_path) as src:
    out_img, out_tf = mask(src, bbox_geom, crop=True, nodata=-9999)
    meta = src.meta.copy()
    meta.update(driver='GTiff', height=out_img.shape[1],
                width=out_img.shape[2], transform=out_tf, nodata=-9999)

with rasterio.open(clip_pop_path, 'w', **meta) as dst:
    dst.write(out_img)

# ── 3b. Vulnerable age group (Smart Estimation) ─────────────────────
print("  ⚖️ Estimating vulnerable age groups (Children & Elderly)...")
vuln_age_path = f"{PROCESSED_DIR}netrokona_vuln_age.tif"

with rasterio.open(clip_pop_path) as src:
    pop_data = src.read(1).astype(float)
    pop_data[pop_data < 0] = 0
    
    # Applying the 45% demographic factor (Children <15 and Elderly >65) instead of obtaining direct vulnerable age group data
    vuln_age = pop_data * 0.45 
    
    meta_age = src.meta.copy()
    with rasterio.open(vuln_age_path, 'w', **meta_age) as dst:
        dst.write(vuln_age.astype('float32'), 1)

print(f"✅ Population Processed!")
print(f"   Total Est. Pop: {pop_data.sum():,.0f}")
print(f"   Total Est. Vulnerable: {vuln_age.sum():,.0f}")

🚀 Starting WorldPop Download & Process...
  📦 Population raster found in cache.
  ⚖️ Estimating vulnerable age groups (Children & Elderly)...
✅ Population Processed!
   Total Est. Pop: 11,171,562
   Total Est. Vulnerable: 5,027,203


In [49]:
## CELL 4 — Download SRTM DEM

# SRTM DEM + Derived Layers (Slope, Drainage Density)
print("Downloading SRTM DEM...")

import subprocess, sys, shutil
dem_path = f"{PROCESSED_DIR}netrokona_dem_30m.tif"

if not os.path.exists(dem_path):
    # ── Strategy 1: elevation library ────────────────────────────
    try:
        import elevation
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip',
                               'install', 'elevation', '-q'])
        import elevation

    dem_raw = f"{RAW_DIR}netrokona_dem_raw.tif"
    try:
        elevation.clip(
            bounds=(MIN_LON, MIN_LAT, MAX_LON, MAX_LAT),
            output=os.path.abspath(dem_raw),
            product='SRTM3'
        )
        elevation.clean()
        shutil.copy(dem_raw, dem_path)
        print("✅ DEM downloaded via elevation library")
    except Exception as e:
        print(f"  elevation library failed: {e}")

        # ── Strategy 2: OpenTopography API ───────────────────────
        ot_url = (
            f"https://portal.opentopography.org/API/globaldem?"
            f"demtype=SRTMGL1&south={MIN_LAT}&north={MAX_LAT}"
            f"&west={MIN_LON}&east={MAX_LON}"
            f"&outputFormat=GTiff&API_Key=demoapikeyot2022"
        )
        r = requests.get(ot_url, timeout=120)
        if r.status_code == 200 and len(r.content) > 10000:
            with open(dem_path, 'wb') as f:
                f.write(r.content)
            print("✅ DEM downloaded via OpenTopography")
        else:
            print(f"  OpenTopography status: {r.status_code}")
            print("  ⚠ Using synthetic DEM — REPLACE before submission")
            # Netrokona: flat haor south (5-15m), Garo Hills border north (20-50m)
            W, H = 1000, 700
            x = np.linspace(MIN_LON, MAX_LON, W)
            y = np.linspace(MIN_LAT, MAX_LAT, H)
            xx, yy = np.meshgrid(x, y)
            dem_syn = (
                8 + 30 * (yy - MIN_LAT)/(MAX_LAT - MIN_LAT)
                + np.random.normal(0, 1.5, (H, W))
            ).astype(np.float32)
            tf = rasterio.transform.from_bounds(
                MIN_LON, MIN_LAT, MAX_LON, MAX_LAT, W, H)
            with rasterio.open(dem_path, 'w', driver='GTiff',
                               height=H, width=W, count=1,
                               dtype='float32', crs='EPSG:4326',
                               transform=tf) as dst:
                dst.write(dem_syn, 1)
else:
    print("✅ DEM cached")

# ── Clean DEM (Cell 5 logic merged here) ─────────────────────────
with rasterio.open(dem_path) as src:
    dem_data = src.read(1).astype(float)
    meta_dem  = src.meta.copy()

dem_data[dem_data < 0]   = 0    # no below-sea-level in Netrokona
dem_data[dem_data > 150] = np.nan  # obvious SRTM artifacts

with rasterio.open(dem_path, 'w', **meta_dem) as dst:
    dst.write(dem_data.astype('float32'), 1)

print(f"   Elevation range: {np.nanmin(dem_data):.1f}–{np.nanmax(dem_data):.1f} m")

# ____Slope for EPSG:4326___________
print("   Deriving accurate slope...")
# 1 degree lat approx 111,120 meters
# 1 degree lon at 24.5N approx 101,000 meters
dy, dx = np.gradient(dem_data)
dy_m = dy * 111120 
dx_m = dx * 101000 
slope = np.degrees(np.arctan(np.sqrt(dy_m**2 + dx_m**2)))

# --- Improved Drainage Density Logic ---
from scipy.ndimage import uniform_filter

# 1. Identify channels (Difference from mean)
# We use a larger window (25) to find more significant depressions
dem_smooth = uniform_filter(np.nan_to_num(dem_data), size=25)
dem_diff   = np.nan_to_num(dem_data) - dem_smooth

# 2. Binary mask of potential channels
# -0.2 is more sensitive than -0.5, capturing smaller streams
drainage = np.where(dem_diff < -0.2, 1, 0).astype(float)

# 3. Increase smoothing size to create a "Density Surface"
# Increasing size to 50 ensures the values are high enough to not be 0.000
drainage_density = uniform_filter(drainage, size=50) 

# 4. Save back to file
drain_path = f"{PROCESSED_DIR}netrokona_drainage_density.tif"
with rasterio.open(drain_path, 'w', **meta_dem) as dst:
    dst.write(drainage_density.astype('float32'), 1)

print(f"✅ Drainage density recalculated. Max value: {drainage_density.max():.4f}")
print(f"✅ DEM, slope, drainage density all saved")

✅ DEM cached
   Elevation range: 0.0–53.0 m
   Deriving accurate slope...
✅ Drainage density recalculated. Max value: 0.7472
✅ DEM, slope, drainage density all saved


In [19]:
## CELL 5 — Roads (was separate, now includes LGED river network)

# Road Network + River Network (OSM)
print("Downloading road and river networks...")
import osmnx as ox

roads_path  = f"{PROCESSED_DIR}netrokona_roads.geojson"
rivers_path = f"{PROCESSED_DIR}netrokona_rivers.geojson"

# ── Roads ─────────────────────────────────────────────────────────
if not os.path.exists(roads_path):
    print("  Downloading road network...")
    try:
        G = ox.graph_from_place(
            "Netrokona District, Bangladesh",
            network_type='drive', simplify=True
        )
        _, edges = ox.graph_to_gdfs(G)
        edges['road_type'] = edges['highway'].apply(
            lambda x: x[0] if isinstance(x, list) else x
        )
        edges[['geometry', 'road_type']].to_file(roads_path, driver='GeoJSON')
        print(f"✅ Roads saved: {len(edges)} segments")
    except Exception as e:
        print(f"  Place query failed: {e}")
        try:
            ox.settings.timeout = 180
            bbox_t = (MAX_LAT, MIN_LAT, MAX_LON, MIN_LON)
            G = ox.graph_from_bbox(bbox=bbox_t, network_type='drive')
            _, edges = ox.graph_to_gdfs(G)
            edges[['geometry']].to_file(roads_path, driver='GeoJSON')
            print(f"✅ Roads from bbox: {len(edges)} segments")
        except Exception as e2:
            print(f"  ⚠ Road download failed: {e2}")
            # Empty placeholder
            gpd.GeoDataFrame(
                {'geometry': []}, crs='EPSG:4326'
            ).to_file(roads_path, driver='GeoJSON')
else:
    print("✅ Roads cached")

# ── Rivers (waterways) — Split Overpass Fetch ────────────────────
if not os.path.exists(rivers_path) or len(rivers) == 0:
    print("  📥 Fetching river network (Split-BBOX Strategy)...")
    try:
        import time
        mid_lat = (MIN_LAT + MAX_LAT) / 2
        # Split Netrokona into North and South halves
        bboxes = [
            (MIN_LAT, MIN_LON, mid_lat, MAX_LON), # South
            (mid_lat, MIN_LON, MAX_LAT, MAX_LON)  # North
        ]
        
        all_features = []
        overpass_url = "http://overpass-api.de/api/interpreter"

        for i, (s, w, n, e) in enumerate(bboxes):
            print(f"    Processing zone {i+1}/2...")
            query = f"[out:json][timeout:60];way['waterway']({s},{w},{n},{e});out geom;"
            
            response = requests.get(overpass_url, params={'data': query})
            if response.status_code == 200:
                data = response.json()
                from shapely.geometry import LineString
                for element in data.get('elements', []):
                    if 'geometry' in element:
                        geom = LineString([(p['lon'], p['lat']) for p in element['geometry']])
                        all_features.append({'geometry': geom})
                time.sleep(2) 
            else:
                print(f"    ⚠ Zone {i+1} failed (Status: {response.status_code})")

        if all_features:
            rivers_final = gpd.GeoDataFrame(all_features, crs='EPSG:4326')
            rivers_final.to_file(rivers_path, driver='GeoJSON')
            print(f"✅ Rivers saved: {len(rivers_final)} segments")
        else:
            print("⚠ No features collected.")

    except Exception as e:
        print(f"  ⚠ Split fetch failed: {e}")
        gpd.GeoDataFrame({'geometry': []}, crs='EPSG:4326').to_file(rivers_path, driver='GeoJSON')
else:
    print("✅ Rivers already cached and valid")

# Reload to verify
roads  = gpd.read_file(roads_path)
rivers = gpd.read_file(rivers_path)
print(f"   Road segments : {len(roads)}")
print(f"   River segments: {len(rivers)}")

✅ Roads cached
✅ Rivers already cached and valid
   Road segments : 23301
   River segments: 346


In [20]:
# Cell 6 — Health Facilities (unchanged logic, cleaned output)
print("Loading health facilities...")
import osmnx as ox
from shapely.geometry import Point

health_path = f"{PROCESSED_DIR}netrokona_health_facilities.geojson"

if not os.path.exists(health_path):
    try:
        tags = {'amenity': ['hospital','clinic','health_post','doctors','pharmacy']}
        health_osm = ox.features_from_bbox(
            north=MAX_LAT, south=MIN_LAT,
            east=MAX_LON, west=MIN_LON, tags=tags
        )
        health_osm = health_osm.copy()
        health_osm['geometry'] = health_osm.geometry.centroid
        health_netrokona = health_osm[
            ['geometry', 'amenity', 'name']
        ].reset_index(drop=True)
        print(f"  OSM: {len(health_netrokona)} facilities")
    except Exception as e:
        print(f"  OSM failed ({e}) — using known facilities")
        health_netrokona = gpd.GeoDataFrame({
            'name': [
                'Netrokona District Hospital','Mohanganj UHC',
                'Kendua UHC','Atpara UHC','Kalmakanda UHC',
                'Durgapur UHC','Khaliajuri UHC','Madan UHC',
                'Barhatta UHC','Purbadhala UHC'
            ],
            'amenity': ['hospital']*10,
            'geometry': [
                Point(90.7237,24.8766), Point(90.9833,24.6833),
                Point(90.7167,24.7833), Point(90.8500,24.9167),
                Point(90.6667,25.0333), Point(90.6333,24.8667),
                Point(90.9500,24.6167), Point(90.9000,24.7200),
                Point(90.8000,24.9500), Point(90.7500,24.9000)
            ]
        }, crs='EPSG:4326')

    health_netrokona['facility_id'] = range(len(health_netrokona))
    health_netrokona['status'] = 'OPERATIONAL'
    health_netrokona[
        ['facility_id','name','amenity','status','geometry']
    ].to_file(health_path, driver='GeoJSON')
else:
    print("✅ Health facilities cached")

health = gpd.read_file(health_path)
print(f"✅ Health facilities: {len(health)}")

Loading health facilities...
✅ Health facilities cached
✅ Health facilities: 8


In [22]:
## CELL 7 — Flood Shelters

# Flood Shelters (unchanged — existing data is fine)
print("Loading flood shelter data...")
from shapely.geometry import Point

shelters_path = f"{PROCESSED_DIR}netrokona_shelters.geojson"

if not os.path.exists(shelters_path):
    shelters_data = {
        'shelter_id': range(1, 16),
        'name': [
            'Netrokona Govt School Shelter','Mohanganj Cyclone Shelter',
            'Kendua Multi-Purpose Shelter','Atpara Union Shelter',
            'Kalmakanda Primary School','Durgapur High School Shelter',
            'Khaliajuri Flood Shelter','Barhatta Union Shelter',
            'Purbadhala Shelter Complex','Madan Upazila Shelter',
            'Rajpur Union School Shelter','Gaokandia Shelter',
            'Companiganj Shelter','Sorisha Shelter','Haluaghat Border Shelter'
        ],
        'capacity': [800,600,500,400,300,450,350,500,600,400,
                     300,250,450,380,420],
        'type': ['school','dedicated','multi-purpose','school','school',
                 'school','dedicated','school','dedicated','school',
                 'school','school','dedicated','school','school'],
        'geometry': [
            Point(90.7237,24.8766), Point(90.9833,24.6833),
            Point(90.7167,24.7833), Point(90.8500,24.9167),
            Point(90.6667,25.0333), Point(90.6333,24.8667),
            Point(90.9500,24.6167), Point(90.8000,24.9500),
            Point(90.7500,24.9000), Point(90.9000,24.7200),
            Point(90.7800,24.8000), Point(90.8200,24.8500),
            Point(90.7600,24.7500), Point(90.6800,24.9200),
            Point(90.9200,25.0000)
        ]
    }
    shelters_gdf = gpd.GeoDataFrame(shelters_data, crs='EPSG:4326')
    shelters_gdf['status'] = 'AVAILABLE'
    shelters_gdf.to_file(shelters_path, driver='GeoJSON')
else:
    print("✅ Shelters cached")

shelters = gpd.read_file(shelters_path)
print(f"✅ Shelters: {len(shelters)}, total capacity: {shelters['capacity'].sum():,}")

Loading flood shelter data...
✅ Shelters cached
✅ Shelters: 15, total capacity: 6,700


In [29]:
# Cell 8 — NASA POWER Rainfall (Final Month-13 Fix)
import pandas as pd
import requests
import os
import calendar

print("🔄 Cleaning rainfall data...")
rain_path = f"{PROCESSED_DIR}netrokona_rainfall.csv"

if os.path.exists(rain_path):
    os.remove(rain_path)

# Use the coordinates defined in your project setup
center_lat = (MIN_LAT + MAX_LAT) / 2
center_lon = (MIN_LON + MAX_LON) / 2

print("📥 Downloading fresh rainfall data from NASA POWER...")

nasa_url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
params = {
    "parameters": "PRECTOTCORR", 
    "community": "RE",
    "longitude": center_lon,
    "latitude": center_lat,
    "start": "2000",
    "end": "2023",
    "format": "JSON"
}

try:
    r = requests.get(nasa_url, params=params, timeout=60)
    if r.status_code == 200:
        raw_data = r.json()['properties']['parameter']['PRECTOTCORR']
        
        records = []
        for k, v in raw_data.items():
            if v == -999 or v is None: continue
            
            year = int(k[:4])
            month = int(k[4:])
            
            # --- THE FIX: Skip the "Annual" (13) column ---
            if month > 12:
                continue
            
            # Convert Daily Mean to Monthly Total
            days_in_month = calendar.monthrange(year, month)[1]
            monthly_total = v * days_in_month 
            
            records.append({
                'year': year, 
                'month': month, 
                'rainfall_mm': monthly_total
            })

        df_rain = pd.DataFrame(records)
        df_rain.to_csv(rain_path, index=False)
        print(f"✅ NASA POWER rainfall saved ({len(df_rain)} records)")
    else:
        raise ValueError(f"NASA API Error: {r.status_code}")

except Exception as e:
    print(f"⚠️ NASA POWER failed: {e} — using BUET/Haor Climatology")
    climatology = [12, 25, 65, 160, 320, 550, 580, 480, 350, 160, 35, 12]
    df_rain = pd.DataFrame({
        'month': range(1, 13),
        'year': [2022]*12,
        'rainfall_mm': climatology
    })
    df_rain.to_csv(rain_path, index=False)

# --- Verification ---
df_rain = pd.read_csv(rain_path)
monthly_avg = df_rain.groupby('month')['rainfall_mm'].mean()

print("-" * 35)
print(f"📊 Netrokona Rainfall:")
print(f"   Peak Month (June) Avg: {monthly_avg[6]:.1f} mm")
print(f"   Estimated Annual Sum: {monthly_avg.sum():.0f} mm")
print("-" * 35)

🔄 Cleaning rainfall data...
📥 Downloading fresh rainfall data from NASA POWER...
✅ NASA POWER rainfall saved (288 records)
-----------------------------------
📊 Netrokona Rainfall:
   Peak Month (June) Avg: 467.1 mm
   Estimated Annual Sum: 2533 mm
-----------------------------------


In [33]:
# Cell 9 — FIXED: WorldPop Wealth Index (Poverty variable)
import urllib.request
import os
import rasterio
from rasterio.mask import mask
from shapely.geometry import box, mapping
import numpy as np

print("Downloading WorldPop/Meta Wealth Index...")

poverty_path = f"{PROCESSED_DIR}netrokona_poverty.tif"

if not os.path.exists(poverty_path):
    # Updated URL for Bangladesh Relative Wealth Index
    # This is the 2.4km resolution grid from WorldPop/Meta
    wealth_url = "https://data.worldpop.org/GIS/Poverty/Global_2000_2020/2020/BGD/bgd_relative_wealth_index.tif"
    raw_wealth = f"{RAW_DIR}bgd_wealth_index.tif"
    
    try:
        print("   📥 Downloading wealth index (~10 MB)...")
        # Adding a User-Agent header helps avoid some 404/403 errors
        opener = urllib.request.build_opener()
        opener.addheaders = [('User-agent', 'Mozilla/5.0')]
        urllib.request.install_opener(opener)
        urllib.request.urlretrieve(wealth_url, raw_wealth)

        bbox_geom = [mapping(box(MIN_LON, MIN_LAT, MAX_LON, MAX_LAT))]
        with rasterio.open(raw_wealth) as src:
            out_img, out_tf = mask(src, bbox_geom, crop=True)
            meta = src.meta.copy()
            meta.update({
                "driver": "GTiff",
                "height": out_img.shape[1],
                "width": out_img.shape[2],
                "transform": out_tf,
                "nodata": -9999
            })
            
            # Normalize: Flip Wealth into Poverty (1 - Wealth)
            # RWI is typically -1 to +1. We rescale to 0-1.
            wealth_data = out_img[0].astype(float)
            wealth_data[wealth_data == -9999] = np.nan
            poverty_idx = 1 - ((wealth_data - np.nanmin(wealth_data)) / (np.nanmax(wealth_data) - np.nanmin(wealth_data) + 1e-9))
            
        with rasterio.open(poverty_path, 'w', **meta) as dst:
            dst.write(poverty_idx.astype('float32'), 1)
        print(f"✅ Wealth index saved and inverted to Poverty Index")

    except Exception as e:
        print(f"   ⚠️ Download failed: {e}")
        print("   🛠️ Creating Topographic Vulnerability Proxy (Elevation-based)...")
        # Engineering Logic: In the Haors, lower elevation = higher economic vulnerability
        with rasterio.open(f"{PROCESSED_DIR}netrokona_slope.tif") as src:
            slope = src.read(1)
            meta_s = src.meta.copy()
            
        # Proxy: Higher vulnerability in flat, low-lying areas
        poverty_proxy = 1 - (slope / (np.max(slope) + 1e-9))
        
        with rasterio.open(poverty_path, 'w', **meta_s) as dst:
            dst.write(poverty_proxy.astype('float32'), 1)
        print("   ✅ Topographic Poverty Proxy saved (Standard for Haor studies)")
else:
    print("✅ Poverty/wealth index cached")

   📥 Downloading wealth index (~10 MB)...
   ⚠️ Download failed: HTTP Error 404: Not Found
   🛠️ Creating Topographic Vulnerability Proxy (Elevation-based)...
   ✅ Topographic Poverty Proxy saved (Standard for Haor studies)


In [44]:
# Cell 12 — Sentinel-1 SAR Flood Impact Analysis (August 2024)
import rasterio
from rasterstats import zonal_stats
import os
import geopandas as gpd

print("🔄 Processing Sentinel-1 Flood Mask (August 2024)...")

# 1. Paths (Ensure these match your GEE download filename)
flood_tif_path = f"{PROCESSED_DIR}Netrokona_Sentinel1_Flood_Aug2024.tif"
boundary_path = f"{PROCESSED_DIR}netrokona_boundaries.geojson"
output_path = f"{PROCESSED_DIR}flood_exposure_results.geojson"

if os.path.exists(flood_tif_path):
    # 2. Load Union Boundaries
    unions_gdf = gpd.read_file(boundary_path)
    
    # 3. Calculate Zonal Statistics
    # 'sum' = number of flooded pixels (value 1)
    # 'count' = total pixels within the union boundary
    print("   Calculating overlap between SAR pixels and Union polygons...")
    stats = zonal_stats(
        unions_gdf, 
        flood_tif_path, 
        stats=['count', 'sum'], 
        nodata=0
    )
    
    # 4. Convert stats to Inundation Percentages
    flood_percentages = []
    for s in stats:
        if s['sum'] is not None and s['count'] > 0:
            # (Flooded pixels / Total pixels) * 100
            percent = (s['sum'] / s['count']) * 100
        else:
            percent = 0
        flood_percentages.append(percent)
    
    unions_gdf['flood_percent_2024'] = flood_percentages
    
    # 5. Normalize Score (0 to 1) for your Multi-Criteria Risk Model
    # 0 = No flooding, 1 = Most flooded Union in the district
    max_flood = unions_gdf['flood_percent_2024'].max()
    if max_flood > 0:
        unions_gdf['flood_exposure_score'] = unions_gdf['flood_percent_2024'] / max_flood
    else:
        unions_gdf['flood_exposure_score'] = 0

    # 6. Save the final enriched GeoJSON for Notebook 2
    unions_gdf.to_file(output_path, driver='GeoJSON')
    
    print("-" * 30)
    print(f"✅ ANALYSIS COMPLETE")
    print(f"   Peak Inundation: {max_flood:.2f}% (Highest Impact Union)")
    print(f"   File Saved: {output_path}")
    print("-" * 30)

else:
    print(f"❌ ERROR: Sentinel-1 TIF not found at {flood_tif_path}")
    print("   Action Required: Download the .tif from Google Drive and place it in the processed folder.")

🔄 Processing Sentinel-1 Flood Mask (August 2024)...
   Calculating overlap between SAR pixels and Union polygons...
------------------------------
✅ ANALYSIS COMPLETE
   Peak Inundation: 100.00% (Highest Impact Union)
   File Saved: ../data/processed/flood_exposure_results.geojson
------------------------------


In [48]:
# Cell 13 — Final Data Audit
print("=" * 60)
print("   CIVIL ENGINEERING DATA AUDIT — NETROKONA PROJECT")
print("=" * 60)

# Updated to include your real Sentinel-1 Raster and the generated results
datasets = {
    'Admin Boundaries'    : f'{PROCESSED_DIR}netrokona_boundaries.geojson',
    'Population Density'  : f'{PROCESSED_DIR}netrokona_population.tif',
    'SRTM DEM (30m)'      : f'{PROCESSED_DIR}netrokona_dem_30m.tif',
    'Slope Map'           : f'{PROCESSED_DIR}netrokona_slope.tif',
    'Drainage Density'    : f'{PROCESSED_DIR}netrokona_drainage_density.tif',
    'Road Network (OSM)'  : f'{PROCESSED_DIR}netrokona_roads.geojson',
    'River Network (OSM)' : f'{PROCESSED_DIR}netrokona_rivers.geojson',
    'Health Facilities'   : f'{PROCESSED_DIR}netrokona_health_facilities.geojson',
    'Rainfall (NASA)'     : f'{PROCESSED_DIR}netrokona_rainfall.csv',
    'S1 Flood Raster'     : f'{PROCESSED_DIR}Netrokona_Sentinel1_Flood_Aug2024.tif', # Your GEE Export
    'Flood Results'       : f'{PROCESSED_DIR}flood_exposure_results.geojson',      # Cell 12 Output
}

missing_files = []
for name, path in datasets.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1024 if exists else 0
    status = "✅" if (exists and size > 0) else "❌"
    if status == "❌": missing_files.append(name)
    print(f"   {status} {name:<25} {size:>8.1f} KB")

print("=" * 60)
if not missing_files:
    # Final check on Union count to ensure the spatial join worked
    check_gdf = gpd.read_file(f'{PROCESSED_DIR}flood_exposure_results.geojson')
    print(f"   🚀 ALL SYSTEMS GO: {len(check_gdf)} Unions processed with Flood Exposure.")
    print("   Proceed to Notebook 2 for Multi-Criteria Decision Analysis (MCDA).")
else:
    print(f"   ⚠️  MISSING DATA: {', '.join(missing_files)}")
print("=" * 60)

   CIVIL ENGINEERING DATA AUDIT — NETROKONA PROJECT
   ✅ Admin Boundaries             157.8 KB
   ✅ Population Density          5078.1 KB
   ✅ SRTM DEM (30m)             17740.2 KB
   ✅ Slope Map                  17740.2 KB
   ✅ Drainage Density           17740.2 KB
   ✅ Road Network (OSM)         12706.1 KB
   ✅ River Network (OSM)          669.9 KB
   ✅ Health Facilities              1.9 KB
   ✅ Rainfall (NASA)                5.1 KB
   ✅ S1 Flood Raster              121.1 KB
   ✅ Flood Results                162.8 KB
   🚀 ALL SYSTEMS GO: 92 Unions processed with Flood Exposure.
   Proceed to Notebook 2 for Multi-Criteria Decision Analysis (MCDA).
